# Project 01 (basic): pathfinding in a maze — from BFS to A\*

**Goal:** you implement breadth-first search (BFS) and A\* for a grid maze and
*see* in a direct comparison why a good heuristic helps so much.

**How to work:** run the cells from top to bottom (`Shift + Enter`).
Cells marked `# TODO` are yours to complete. If you get stuck: hints are given
directly in the cell, and the complete reference solution is in
`solution/solution.ipynb` — try it yourself first!

**Reference to the script:** section 1.3 (problem solving by search) in the module README.

## 1. The world: a grid maze

We represent the maze as a list of strings — easy to read and easy to change:

- `#` = wall, `.` = free cell, `S` = start, `G` = goal

A **state** is simply a position `(row, col)`. The **actions** are steps
up/down/left/right (no diagonals), each costing 1.

In [ ]:
MAZE = [
    "####################",
    "#S...#........#....#",
    "#.##.#.######.#.##.#",
    "#.#..#......#.#..#.#",
    "#.#.####.##.#.##.#.#",
    "#.#....#.#..#....#.#",
    "#.####.#.#.#######.#",
    "#......#.#.......#.#",
    "######.#.#######.#.#",
    "#....#.#.......#.#.#",
    "#.##.#.#######.#.#.#",
    "#.#..#.......#.#.#.#",
    "#.#.########.#.#.#.#",
    "#.#...............G#",
    "####################",
]

def parse_maze(lines):
    """Returns (walls, start, goal); walls is a set of (row, col) positions."""
    walls, start, goal = set(), None, None
    for r, line in enumerate(lines):
        for c, cell in enumerate(line):
            if cell == "#":
                walls.add((r, c))
            elif cell == "S":
                start = (r, c)
            elif cell == "G":
                goal = (r, c)
    return walls, start, goal

WALLS, START, GOAL = parse_maze(MAZE)
HEIGHT, WIDTH = len(MAZE), len(MAZE[0])
print(f"Maze {HEIGHT}x{WIDTH}, start {START}, goal {GOAL}, {len(WALLS)} wall cells")

## 2. The transition model: neighbouring cells

First task: write the function `neighbours(pos)` that returns all **walkable**
neighbouring cells of a position (up, down, left, right — but no walls and
nothing outside the grid).

In [ ]:
def neighbours(pos):
    """All walkable neighbouring positions of pos = (row, col)."""
    r, c = pos
    result = []
    # TODO: go through the 4 directions (-1,0),(1,0),(0,-1),(0,1).
    # TODO: check for each: is the cell inside the grid AND not in WALLS?
    # TODO: if so, append it to `result`.
    return result

# Mini test — both lines must print True:
print(sorted(neighbours(START)) == [(1, 2), (2, 1)])   # start: free to the right and below
print((13, 17) in neighbours(GOAL))                    # the cell left of the goal is free

## 3. Breadth-first search (BFS)

BFS expands nodes in the order they are discovered (FIFO queue) and therefore
finds a **shortest** path with certainty (every step costs 1).

For every visited node we remember its **parent**, so that we can walk the path
backwards at the end. We also count the **expanded nodes** — our comparison
measure for later.

In [ ]:
from collections import deque

def reconstruct_path(parent, goal):
    """Walks the parent chain from the goal back to the start."""
    path, pos = [], goal
    while pos is not None:
        path.append(pos)
        pos = parent[pos]
    return path[::-1]

def bfs(start, goal):
    """Returns (path, expanded_nodes)."""
    frontier = deque([start])
    parent = {start: None}
    expanded = []
    while frontier:
        # TODO: 1) take the front element from frontier (popleft) and append it to expanded
        # TODO: 2) if it is the goal: return (reconstruct_path(...), expanded)
        # TODO: 3) otherwise: every neighbours(pos) that is NOT yet in parent
        # TODO:    gets entered into parent (value: pos) and appended to frontier
    return None, expanded  # no path found

path_bfs, exp_bfs = bfs(START, GOAL)
print(f"BFS: path length {len(path_bfs) - 1} steps, {len(exp_bfs)} nodes expanded")

## 4. Making it visible

A small drawing function (given, just run it): walls black, expanded nodes light
blue, the path found in red.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

def draw(expanded, path, title):
    grid = np.zeros((HEIGHT, WIDTH))           # 0 free
    for w in WALLS: grid[w] = 1                # 1 wall
    for p in expanded: grid[p] = 2             # 2 expanded
    for p in path: grid[p] = 3                 # 3 path
    grid[START], grid[GOAL] = 4, 4             # 4 start/goal
    colours = ListedColormap(["white", "black", "#a6cee3", "#e31a1c", "#33a02c"])
    plt.figure(figsize=(7, 5))
    plt.imshow(grid, cmap=colours, vmin=0, vmax=4)
    plt.title(f"{title} — path: {len(path)-1} steps, expanded: {len(expanded)}")
    plt.xticks([]); plt.yticks([])
    plt.show()

draw(exp_bfs, path_bfs, "Breadth-first search (BFS)")

## 5. A\*: search with a sense of direction

BFS spreads out blindly in all directions. A\* uses a **heuristic** $h(n)$
(estimated remaining cost to the goal) and always expands the node with the
smallest

$$f(n) = g(n) + h(n)$$

($g$ = cost so far). For grids without diagonals the **Manhattan distance** is
the appropriate admissible heuristic:

$$h((r,c)) = |r - r_{goal}| + |c - c_{goal}|$$

It never overestimates (each step reduces the distance by at most 1), so A\* is
guaranteed to find the optimal path.

For the priority queue we use Python's `heapq`: a heap from which `heappop`
always takes the smallest element. We put tuples `(f, position)` into it.

In [ ]:
import heapq

def manhattan(pos, goal):
    # TODO: return |dr| + |dc| (see the formula above)

def a_star(start, goal, h):
    """A* search. h is the heuristic function h(pos, goal). Returns (path, expanded)."""
    frontier = [(h(start, goal), start)]        # heap of (f, position)
    parent = {start: None}
    g = {start: 0}                              # cost so far per node
    expanded = []
    while frontier:
        _, pos = heapq.heappop(frontier)
        if pos in expanded:                     # stale heap entry
            continue
        expanded.append(pos)
        if pos == goal:
            return reconstruct_path(parent, goal), expanded
        for n in neighbours(pos):
            # TODO: g_new = g[pos] + 1
            # TODO: if n is unknown OR g_new < g[n]:
            # TODO:   update g[n] and parent[n], and push
            # TODO:   (g_new + h(n, goal), n) onto frontier with heapq.heappush
    return None, expanded

path_astar, exp_astar = a_star(START, GOAL, manhattan)
print(f"A*:  path length {len(path_astar) - 1} steps, {len(exp_astar)} nodes expanded")
draw(exp_astar, path_astar, "A* with Manhattan heuristic")

## 6. The comparison

Both find an equally long (optimal) path — but look at the number of expanded
nodes! To finish, two more experiments:

1. **Greedy best-first** = A\* without $g$: we pass in a "heuristic" but do not
   count $g$. We simulate this by putting a function into `a_star` that weights
   the Manhattan distance heavily — the heavier, the greedier.
2. **Zero heuristic** $h = 0$: then $f = g$ and A\* becomes uniform-cost search
   (here identical to the behaviour of BFS).

In [ ]:
# Experiment: comparing different heuristics
candidates = {
    "h = 0 (uniform cost / like BFS)": lambda p, g: 0,
    "h = Manhattan (A*)":              manhattan,
    "h = 5 * Manhattan (greedy!)":     lambda p, g: 5 * manhattan(p, g),
}
print(f"{'Heuristic':<35}{'path length':>13}{'expanded':>12}")
for name, h in candidates.items():
    path, exp = a_star(START, GOAL, h)
    print(f"{name:<35}{len(path) - 1:>13}{len(exp):>12}")

**Observation:** the greedy variant (inflated heuristic) expands the fewest
nodes — but depending on the maze it finds a **longer** path, because an
overestimating heuristic destroys the optimality guarantee (script, section 1.3).
Manhattan is the sweet spot: optimal *and* markedly more economical than blind
search.

## Done — what you can do now

- formulate a problem as a state space (states, actions, goal, costs)
- implement BFS and A\* and reconstruct the path
- explain and *show* what admissibility of a heuristic means in practice

**Bonus tasks** (optional, no reference solution):
1. Add weights: `~` cells (swamp) cost 5 instead of 1. Where does the code have to change?
2. Allow diagonal steps (cost $\sqrt{2}$). Why is Manhattan then NOT admissible any more, and which heuristic fits instead?
3. Use `%timeit` to measure the runtime of BFS vs. A\* on a larger random maze.